# 星系形态分类数据集

In [ ]:
import pandas as pd
import json
import os

def convert_class_files_with_features():
    # ================= 配置参数 =================
    DATA_DIR = "/remote-home/cs_acmis_hby/AstroData"
    INPUT_CSV = os.path.join(DATA_DIR, "parameter_features.csv")
    IMAGE_BASE = f"{DATA_DIR}/images_gz2/images"
    OUTPUT_JSON = "/remote-home/cs_acmis_hby/code/FT-LLM/galaxy_classification/galaxy_Para_with_Morphology_features.json"
    
    # ================= 类型映射 =================
    GALAXY_TYPES = {
        0: ("round elliptical", "A"),
        1: ("in-between elliptical", "B"),
        2: ("cigar-shaped elliptical", "C"),
        3: ("edge-on", "D"),
        4: ("Barred spirals", "E"),
        5: ("Unbarred spirals", "F"),
        6: ("Irregular", "G"),
        7: ("merger", "H")
    }
    
    # ================= 特征列映射 =================
    FEATURE_MAPPING = {
        't01_1': 'f_smooth',
        't01_2': 'f_features/disk',
        't02_1': 'f_edge-on/yes',
        't02_2': 'f_edge-on/no',
        't03_1': 'f_bar/yes',
        't03_2': 'f_bar/no',
        't04_1': 'f_spiral/yes',
        't07_1': 'f_completelyround',
        't07_2': 'f_in-between',
        't07_3': 'f_cigar-shaped',
        't06_1': 'f_odd/yes',
        't06_2': 'f_odd/no',
        't08_3': 'f_disturbed',
        't08_4': 'f_irregular',
        't08_5': 'f_other',
        't08_6': 'f_merger',
        't08_7': 'f_dustlane'
    }

    # ================= 分类阈值标准 =================
    CLASS_CRITERIA = {
        0: ["f_smooth >= 0.469", "f_completelyround >= 0.5", "f_odd/no >= 0.5"],
        1: ["f_smooth >= 0.469", "f_in-between >= 0.5", "f_odd/no >= 0.5"],
        2: ["f_smooth >= 0.469", "f_cigar-shaped >= 0.5", "f_odd/no >= 0.5"],
        3: ["f_features/disk >= 0.430", "f_edge-on/yes >= 0.602", "f_odd/no >= 0.5"],
        4: ["f_features/disk >= 0.430", "f_edge-on/no >= 0.715", "f_bar/yes >= 0.715", "f_spiral/yes >= 0.619"],
        5: ["f_features/disk >= 0.430", "f_edge-on/no >= 0.715", "f_bar/no >= 0.715", "f_spiral/yes >= 0.619"],
        6: ["f_odd/yes >= 0.420", "(f_disturbed >= 0.5 OR f_irregular >= 0.5 OR f_other >= 0.5 OR f_dustlane >= 0.5)"],
        7: ["f_odd/yes >= 0.420", "f_merger >= 0.5"]
    }

    # ================= 主处理逻辑 =================
    try:
        # 读取数据并重命名列
        df = pd.read_csv(
            INPUT_CSV,
            dtype={'galaxyID': 'Int64'},
            usecols=list(FEATURE_MAPPING.keys()) + ['galaxyID', 'label1']
        ).rename(columns=FEATURE_MAPPING)

        print(f"✅ 成功读取数据文件，总样本数: {len(df):,}")

        # 校验数据完整性
        required_cols = list(FEATURE_MAPPING.values()) + ['galaxyID', 'label1']
        if missing := [col for col in required_cols if col not in df.columns]:
            raise KeyError(f"缺失关键列: {missing}")

        # 清洗galaxyID
        if not df['galaxyID'].apply(lambda x: x == int(x)).all():
            invalid = df[df['galaxyID'] != df['galaxyID'].astype(int)]['galaxyID']
            print(f"⚠️ 警告: 发现{len(invalid)}个非整型ID，执行取整操作")
            df['galaxyID'] = df['galaxyID'].astype(int)

        # 初始化数据结构
        dataset = []
        stats = {k: {"count":0, "type":v[0]} for k,v in GALAXY_TYPES.items()}

        for class_id, (class_name, option_char) in GALAXY_TYPES.items():
            class_df = df[df['label1'] == class_id]
            if class_df.empty:
                print(f"⚠️ 警告: 类别{class_id}({class_name})无数据")
                continue

            def create_conversation(row):
                # 生成特征字符串
                features = ",\n".join([f"{k}={v:.5f}" for k,v in row.items() if k in FEATURE_MAPPING.values()])
                
                # 生成选项列表（严格遵循格式）
                options = ",\n".join([f"'{v[1]}:{v[0]}'" for v in GALAXY_TYPES.values()])
                
                # 获取满足的条件
                met_conditions = []
                for condition in CLASS_CRITERIA[class_id]:
                    if "OR" in condition:
                        parts = [c.strip() for c in condition.strip("()").split("OR")]
                        for part in parts:
                            var, val = part.split(">=")
                            if row[var.strip()] >= float(val.strip()):
                                met_conditions.append(condition.replace(" OR ", " or "))
                                break
                    else:
                        var, val = condition.split(">=")
                        if row[var.strip()] >= float(val.strip()):
                            met_conditions.append(condition)

                return {
                    "id": int(row["galaxyID"]),
                    "image": [f"{IMAGE_BASE}/{int(row['galaxyID'])}.jpg"],
                    "conversations": [
                        {
                            "from": "human",
                            "value": (
                                # f"<image>Given galaxy image and the corresponding physical parameters:\n"
                                f"<image>Given galaxy image.\n"
                                f"[Output Constraints] Return the answer as one of the following options:\n"
                                f"{options}\n\n"
                                "Now, based on the morphological image, return the choice."
                            )
                        },
                        {
                            "from": "gpt",
                            "value": (
                                "Answer:\n"
                                # "Based on the image information, and the corresponding feature parameters of the image:\n"
                                # f"{'; '.join(met_conditions)}\n"
                                f"Based on the image information, option {option_char} is selected."
                            )
                        }
                    ]
                }

            converted = class_df.apply(create_conversation, axis=1).tolist()
            stats[class_id]["count"] = len(converted)
            dataset.extend(converted)
            print(f"🔄 转换完成 {class_name.ljust(22)}: {len(converted)} 条")

        # 保存结果
        with open(OUTPUT_JSON, 'w') as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)

        # 生成报告
        print("\n▄▄▄▄▄▄▄▄▄ 转换统计 ▄▄▄▄▄▄▄▄▄")
        total = sum(info["count"] for info in stats.values())
        for k in sorted(stats.keys()):
            print(f"类别 {k}: {stats[k]['type'].ljust(24)} {stats[k]['count']:>5} 条")
        print(f"总转换数量: {total} 条")
        print(f"输出文件: {os.path.abspath(OUTPUT_JSON)}")

    except Exception as e:
        print(f"❌ 严重错误: {type(e).__name__} - {str(e)}")

if __name__ == "__main__":
    convert_class_files_with_features()

In [ ]:
import json
import os
import re
import numpy as np
from sklearn.model_selection import train_test_split
from collections import defaultdict, Counter
from PIL import Image

def process_galaxy_data():
    # ---------------------- 配置路径 ----------------------
    json_path = "/remote-home/cs_acmis_hby/code/FT-LLM/galaxy_classification/galaxy_Para_with_Morphology_features.json"
    original_img_dir = "/remote-home/cs_acmis_hby/AstroData/images_gz2/images"
    output_dir = "/remote-home/cs_acmis_hby/code/FT-LLM/galaxy_classification/dataset_07"
    
    # 创建输出目录
    train_img_dir = os.path.join(output_dir, "train_images")
    test_img_dir = os.path.join(output_dir, "test_images")
    os.makedirs(train_img_dir, exist_ok=True)
    os.makedirs(test_img_dir, exist_ok=True)

    # ---------------------- 数据加载与清洗 ----------------------
    print("⏳ 正在加载原始数据...")
    try:
        with open(json_path, 'r') as f:
            raw_data = json.load(f)
        print(f"✅ 成功加载 {len(raw_data)} 条原始数据")
    except Exception as e:
        raise RuntimeError(f"数据加载失败: {str(e)}")

    # 改进的标签提取正则表达式
    LABEL_PATTERN = re.compile(r"option\s*([A-H])", re.IGNORECASE)
    
    def extract_label(content):
        """改进的标签解析函数"""
        match = LABEL_PATTERN.search(content)
        if match:
            return match.group(1).upper()
        # 备用解析逻辑
        if "option A" in content: return "A"
        if "option B" in content: return "B"
        return None

    # 构建格式化数据
    formatted_data = []
    label_counter = Counter()
    missing_images = 0
    label_parse_errors = 0

    print("\n🔍 正在进行数据清洗...")
    for idx, item in enumerate(raw_data):
        try:
            # 验证图像存在性
            valid_images = []
            for img_rel_path in item["image"]:
                img_name = os.path.basename(img_rel_path)
                src_path = os.path.join(original_img_dir, img_name)
                
                if os.path.exists(src_path):
                    valid_images.append(img_name)
                else:
                    missing_images += 1
                    if idx < 5:  # 打印前5个缺失样本
                        print(f"  警告：图像缺失 {img_name}")

            if not valid_images:
                continue

            # 构建对话消息
            messages = []
            for conv in item["conversations"]:
                role = "user" if conv["from"] == "human" else "assistant"
                content = conv["value"]
                
                if role == "user":
                    content = "<image>" + content.split("\n", 1)[-1].strip()
                
                messages.append({
                    "role": role,
                    "content": content.strip()
                })

            # 提取标签
            assistant_content = next(msg["content"] for msg in messages if msg["role"] == "assistant")
            label = extract_label(assistant_content)
            
            if not label:
                label_parse_errors += 1
                if label_parse_errors <= 3:  # 打印前3个解析错误
                    print(f"  标签解析失败: {assistant_content[:50]}...")
                continue
                
            label_counter.update([label])
            formatted_data.append({
                "messages": messages,
                "images": valid_images.copy()
            })

        except Exception as e:
            print(f"⚠️ 数据处理异常（条目{idx}）: {str(e)}")
            continue

    # ---------------------- 数据验证 ----------------------
    print("\n📊 数据清洗结果:")
    print(f" - 有效数据条目: {len(formatted_data)}")
    print(f" - 缺失图像数: {missing_images}")
    print(f" - 标签解析失败数: {label_parse_errors}")
    print(" - 类别分布:")
    for label, count in label_counter.most_common():
        print(f"   {label}: {count} 样本")

    if not label_counter:
        raise ValueError("⚠️ 所有标签解析失败，请检查数据格式！")

    # ---------------------- 分层抽样 ----------------------
    min_samples = 1  # 允许单个样本类别
    valid_labels = [label for label, count in label_counter.items() if count >= min_samples]
    
    print(f"\n🔖 有效类别（≥{min_samples}样本）: {valid_labels}")
    if not valid_labels:
        raise ValueError(f"⚠️ 没有满足最小样本数的类别（当前min_samples={min_samples}）")

    # 构建有效数据索引
    valid_data = []
    label_indices = defaultdict(list)
    for idx, item in enumerate(formatted_data):
        assistant_content = next(msg["content"] for msg in item["messages"] if msg["role"] == "assistant")
        if (label := extract_label(assistant_content)) in valid_labels:
            label_indices[label].append(idx)
            valid_data.append(item)

    # 执行安全抽样
    sampled_indices = []
    print("\n🎲 正在进行分层抽样...")
    for label, indices in label_indices.items():
        sample_size = min(len(indices), 8000)
        if len(indices) < min_samples:
            print(f"  警告：类别 {label} 样本不足（{len(indices)}），已跳过")
            continue
            
        print(f"  {label}: 抽样 {sample_size}/{len(indices)}")
        sampled = np.random.choice(indices, size=sample_size, replace=False)
        sampled_indices.extend(sampled.tolist())
    
    sampled_data = [valid_data[i] for i in sampled_indices]
    print(f"\n📦 抽样后数据量: {len(sampled_data)}")

    # ---------------------- 数据集划分 ----------------------
    print("\n✂️ 正在划分数据集...")
    labels = [extract_label(next(msg["content"] for msg in item["messages"] if msg["role"] == "assistant")) 
             for item in sampled_data]

    # 动态调整参数
    test_size = 0.2 if len(sampled_data) >= 10 else 0.1
    stratify = labels if len(set(labels)) > 1 else None
    
    try:
        train_data, test_data = train_test_split(
            sampled_data,
            test_size=test_size,
            stratify=stratify,
            random_state=42
        )
    except ValueError as e:
        print(f"⚠️ 分层划分失败: {str(e)}")
        print("  尝试非分层划分...")
        train_data, test_data = train_test_split(
            sampled_data,
            test_size=test_size,
            stratify=None,
            random_state=42
        )

    print(f" 训练集: {len(train_data)} | 测试集: {len(test_data)}")

    # ---------------------- 图像处理 ----------------------
    def process_and_copy_images(data, target_img_dir):
        """安全图像处理器"""
        processed_data = []
        for item in data:
            new_images = []
            for img_name in item["images"]:
                src_path = os.path.join(original_img_dir, img_name)
                dst_path = os.path.join(target_img_dir, img_name)
                
                try:
                    with Image.open(src_path) as img:
                        # 中心裁剪
                        w, h = img.size
                        crop_box = (
                            (w - 224) // 2,
                            (h - 224) // 2,
                            (w + 224) // 2,
                            (h + 224) // 2
                        )
                        cropped = img.crop(crop_box)
                        
                        if cropped.mode != 'RGB':
                            cropped = cropped.convert('RGB')
                            
                        cropped.save(dst_path)
                        new_images.append(dst_path)
                except Exception as e:
                    print(f"⚠️ 图像处理失败: {img_name} -> {str(e)}")
                    continue
            
            if new_images:
                processed_data.append({
                    "messages": [
                        {
                            "role": msg["role"],
                            "content": msg["content"].replace("\n", " ").strip()
                        } for msg in item["messages"]
                    ],
                    "images": new_images
                })
        return processed_data

    print("\n🖼️ 正在处理图像...")
    print("  训练集图像处理中...")
    train_processed = process_and_copy_images(train_data, train_img_dir)
    print("  测试集图像处理中...")
    test_processed = process_and_copy_images(test_data, test_img_dir)

    # ---------------------- 数据保存 ----------------------
    def save_jsonl(data, path):
        with open(path, 'w') as f:
            for item in data:
                # 宽松格式验证
                try:
                    assert len(item["messages"]) >= 1, "消息数量不足"
                    assert any(msg["role"] == "user" for msg in item["messages"]), "缺少用户消息"
                    assert any(msg["role"] == "assistant" for msg in item["messages"]), "缺少助手消息"
                except AssertionError as e:
                    print(f"⚠️ 数据格式异常: {e}")
                    continue
                    
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("\n💾 正在保存数据集...")
    save_jsonl(train_processed, os.path.join(output_dir, "train.jsonl"))
    save_jsonl(test_processed, os.path.join(output_dir, "test.jsonl"))

    # ---------------------- 最终报告 ----------------------
    print("\n" + "="*40)
    print("🏁 处理完成！最终统计:")
    print(f" - 原始数据量: {len(raw_data)}")
    print(f" - 有效数据量: {len(formatted_data)}")
    print(f" - 训练集数量: {len(train_processed)}")
    print(f" - 测试集数量: {len(test_processed)}")
    
    if train_processed:
        sample = train_processed[0]
        print("\n🔍 示例样本:")
        print("用户消息:", sample["messages"][0]["content"][:80] + "...")
        print("助手消息:", sample["messages"][1]["content"][:80] + "...")
        print("图像路径:", sample["images"][0])
    else:
        print("\n⚠️ 警告：训练集为空！")

if __name__ == "__main__":
    process_galaxy_data()

# 星系形态属性预测数据集

In [ ]:
import pandas as pd
import json
import os

def convert_class_files_with_features():
    # ================= 配置参数 =================
    DATA_DIR = "/remote-home/cs_acmis_hby/AstroData"
    INPUT_CSV = os.path.join(DATA_DIR, "parameter_features.csv")
    IMAGE_BASE = f"{DATA_DIR}/images_gz2/images"
    OUTPUT_JSON = "/remote-home/cs_acmis_hby/code/FT-LLM/galaxy_classification/galaxy_Para_with_Attribute_features.json"
    
    # ================= 类型映射 =================
    GALAXY_TYPES = {
        0: ("round elliptical", "A"),
        1: ("in-between elliptical", "B"),
        2: ("cigar-shaped elliptical", "C"),
        3: ("edge-on", "D"),
        4: ("Barred spirals", "E"),
        5: ("Unbarred spirals", "F"),
        6: ("Irregular", "G"),
        7: ("merger", "H")
    }
    
    # ================= 特征列映射 =================
    FEATURE_MAPPING = {
        't01_1': 'f_smooth',
        't01_2': 'f_features/disk',
        't02_1': 'f_edge-on/yes',
        't02_2': 'f_edge-on/no',
        't03_1': 'f_bar/yes',
        't03_2': 'f_bar/no',
        't04_1': 'f_spiral/yes',
        't07_1': 'f_completelyround',
        't07_2': 'f_in-between',
        't07_3': 'f_cigar-shaped',
        't06_1': 'f_odd/yes',
        't06_2': 'f_odd/no',
        't08_3': 'f_disturbed',
        't08_4': 'f_irregular',
        't08_5': 'f_other',
        't08_6': 'f_merger',
        't08_7': 'f_dustlane'
        
    }

    # ================= 分类阈值标准 =================
    CLASS_CRITERIA = {
        0: ["f_smooth >= 0.469", "f_completelyround >= 0.5", "f_odd/no >= 0.5"],
        1: ["f_smooth >= 0.469", "f_in-between >= 0.5", "f_odd/no >= 0.5"],
        2: ["f_smooth >= 0.469", "f_cigar-shaped >= 0.5", "f_odd/no >= 0.5"],
        3: ["f_features/disk >= 0.430", "f_edge-on/yes >= 0.602", "f_odd/no >= 0.5"],
        4: ["f_features/disk >= 0.430", "f_edge-on/no >= 0.715", "f_bar/yes >= 0.715", "f_spiral/yes >= 0.619"],
        5: ["f_features/disk >= 0.430", "f_edge-on/no >= 0.715", "f_bar/no >= 0.715", "f_spiral/yes >= 0.619"],
        6: ["f_odd/yes >= 0.420", "(f_disturbed >= 0.5 OR f_irregular >= 0.5 OR f_other >= 0.5 OR f_dustlane >= 0.5)"],
        7: ["f_odd/yes >= 0.420", "f_merger >= 0.5"]
    }

    # ================= 主处理逻辑 =================
    try:
        # 读取数据并重命名列
        df = pd.read_csv(
            INPUT_CSV,
            dtype={'galaxyID': 'Int64'},
            usecols=list(FEATURE_MAPPING.keys()) + ['galaxyID', 'label1']
        ).rename(columns=FEATURE_MAPPING)

        print(f"✅ 成功读取数据文件，总样本数: {len(df):,}")

        # 校验数据完整性
        required_cols = list(FEATURE_MAPPING.values()) + ['galaxyID', 'label1']
        if missing := [col for col in required_cols if col not in df.columns]:
            raise KeyError(f"缺失关键列: {missing}")

        # 清洗galaxyID
        if not df['galaxyID'].apply(lambda x: x == int(x)).all():
            invalid = df[df['galaxyID'] != df['galaxyID'].astype(int)]['galaxyID']
            print(f"⚠️ 警告: 发现{len(invalid)}个非整型ID，执行取整操作")
            df['galaxyID'] = df['galaxyID'].astype(int)

        # 初始化数据结构
        dataset = []
        stats = {k: {"count":0, "type":v[0]} for k,v in GALAXY_TYPES.items()}

        for class_id, (class_name, option_char) in GALAXY_TYPES.items():
            class_df = df[df['label1'] == class_id]
            if class_df.empty:
                print(f"⚠️ 警告: 类别{class_id}({class_name})无数据")
                continue

            def create_conversation(row):
                # 生成特征字符串
                features = ",\n".join([f"{k}={v:.5f}" for k,v in row.items() if k in FEATURE_MAPPING.values()])
                
                # 生成选项列表（严格遵循格式）
                options = ",\n".join([f"'{v[1]}:{v[0]}'" for v in GALAXY_TYPES.values()])
                
                # 获取满足的条件
                met_conditions = []
                for condition in CLASS_CRITERIA[class_id]:
                    if "OR" in condition:
                        parts = [c.strip() for c in condition.strip("()").split("OR")]
                        for part in parts:
                            var, val = part.split(">=")
                            if row[var.strip()] >= float(val.strip()):
                                met_conditions.append(condition.replace(" OR ", " or "))
                                break
                    else:
                        var, val = condition.split(">=")
                        if row[var.strip()] >= float(val.strip()):
                            met_conditions.append(condition)

                return {
                    "id": int(row["galaxyID"]),
                    "image": [f"{IMAGE_BASE}/{int(row['galaxyID'])}.jpg"],
                    "conversations": [
                        {
                            "from": "human",
                            "value": (
                                f"<image>Given galaxy image:\n"
                                # f"{features}\n\n"
                                # f"[Output Constraints] Return the answer as one of the following options:\n"
                                f"{options}\n"
                                f"Galaxy morphology is option {option_char}.\n"
                                "Now, based on the choice and image, return the features of the galaxy  morphology."
                            )
                        },
                        {
                            "from": "gpt",
                            "value": (
                                "Answer:\n"
                                f"Based on the image information and the option {option_char} is selected.\n"
                                # f"{'; '.join(met_conditions)}\n" option {option_char} is selected
                                f"Therefore, the corresponding feature parameters of the image are:\n"
                                f"{features}\n\n"  
                            )
                        }
                    ]
                }

            converted = class_df.apply(create_conversation, axis=1).tolist()
            stats[class_id]["count"] = len(converted)
            dataset.extend(converted)
            print(f"🔄 转换完成 {class_name.ljust(22)}: {len(converted)} 条")

        # 保存结果
        with open(OUTPUT_JSON, 'w') as f:
            json.dump(dataset, f, indent=2, ensure_ascii=False)

        # 生成报告
        print("\n▄▄▄▄▄▄▄▄▄ 转换统计 ▄▄▄▄▄▄▄▄▄")
        total = sum(info["count"] for info in stats.values())
        for k in sorted(stats.keys()):
            print(f"类别 {k}: {stats[k]['type'].ljust(24)} {stats[k]['count']:>5} 条")
        print(f"总转换数量: {total} 条")
        print(f"输出文件: {os.path.abspath(OUTPUT_JSON)}")

    except Exception as e:
        print(f"❌ 严重错误: {type(e).__name__} - {str(e)}")

if __name__ == "__main__":
    convert_class_files_with_features()

In [ ]:
import json
import os
import re
import numpy as np
from sklearn.model_selection import train_test_split
from collections import defaultdict, Counter
from PIL import Image

def process_galaxy_data():
    # ---------------------- 配置路径 ----------------------
    json_path = "/remote-home/cs_acmis_hby/code/FT-LLM/galaxy_classification/galaxy_Para_Attribute_features.json"
    original_img_dir = "/remote-home/cs_acmis_hby/AstroData/images_gz2/images"
    output_dir = "/remote-home/cs_acmis_hby/code/FT-LLM/galaxy_classification/dataset_05"
    
    # 创建输出目录
    train_img_dir = os.path.join(output_dir, "train_images")
    test_img_dir = os.path.join(output_dir, "test_images")
    os.makedirs(train_img_dir, exist_ok=True)
    os.makedirs(test_img_dir, exist_ok=True)

    # ---------------------- 数据加载与清洗 ----------------------
    print("⏳ 正在加载原始数据...")
    try:
        with open(json_path, 'r') as f:
            raw_data = json.load(f)
        print(f"✅ 成功加载 {len(raw_data)} 条原始数据")
    except Exception as e:
        raise RuntimeError(f"数据加载失败: {str(e)}")

    # 改进的标签提取正则表达式
    LABEL_PATTERN = re.compile(r"option\s*([A-H])", re.IGNORECASE)
    
    def extract_label(content):
        """改进的标签解析函数"""
        match = LABEL_PATTERN.search(content)
        if match:
            return match.group(1).upper()
        # 备用解析逻辑
        if "option A" in content: return "A"
        if "option B" in content: return "B"
        return None

    # 构建格式化数据
    formatted_data = []
    label_counter = Counter()
    missing_images = 0
    label_parse_errors = 0

    print("\n🔍 正在进行数据清洗...")
    for idx, item in enumerate(raw_data):
        try:
            # 验证图像存在性
            valid_images = []
            for img_rel_path in item["image"]:
                img_name = os.path.basename(img_rel_path)
                src_path = os.path.join(original_img_dir, img_name)
                
                if os.path.exists(src_path):
                    valid_images.append(img_name)
                else:
                    missing_images += 1
                    if idx < 5:  # 打印前5个缺失样本
                        print(f"  警告：图像缺失 {img_name}")

            if not valid_images:
                continue

            # 构建对话消息
            messages = []
            for conv in item["conversations"]:
                role = "user" if conv["from"] == "human" else "assistant"
                content = conv["value"]
                
                if role == "user":
                    content = "<image>" + content.split("\n", 1)[-1].strip()
                
                messages.append({
                    "role": role,
                    "content": content.strip()
                })

            # 提取标签
            assistant_content = next(msg["content"] for msg in messages if msg["role"] == "assistant")
            label = extract_label(assistant_content)
            
            if not label:
                label_parse_errors += 1
                if label_parse_errors <= 3:  # 打印前3个解析错误
                    print(f"  标签解析失败: {assistant_content[:50]}...")
                continue
                
            label_counter.update([label])
            formatted_data.append({
                "messages": messages,
                "images": valid_images.copy()
            })

        except Exception as e:
            print(f"⚠️ 数据处理异常（条目{idx}）: {str(e)}")
            continue

    # ---------------------- 数据验证 ----------------------
    print("\n📊 数据清洗结果:")
    print(f" - 有效数据条目: {len(formatted_data)}")
    print(f" - 缺失图像数: {missing_images}")
    print(f" - 标签解析失败数: {label_parse_errors}")
    print(" - 类别分布:")
    for label, count in label_counter.most_common():
        print(f"   {label}: {count} 样本")

    if not label_counter:
        raise ValueError("⚠️ 所有标签解析失败，请检查数据格式！")

    # ---------------------- 分层抽样 ----------------------
    min_samples = 1  # 允许单个样本类别
    valid_labels = [label for label, count in label_counter.items() if count >= min_samples]
    
    print(f"\n🔖 有效类别（≥{min_samples}样本）: {valid_labels}")
    if not valid_labels:
        raise ValueError(f"⚠️ 没有满足最小样本数的类别（当前min_samples={min_samples}）")

    # 构建有效数据索引
    valid_data = []
    label_indices = defaultdict(list)
    for idx, item in enumerate(formatted_data):
        assistant_content = next(msg["content"] for msg in item["messages"] if msg["role"] == "assistant")
        if (label := extract_label(assistant_content)) in valid_labels:
            label_indices[label].append(idx)
            valid_data.append(item)

    # 执行安全抽样
    sampled_indices = []
    print("\n🎲 正在进行分层抽样...")
    for label, indices in label_indices.items():
        sample_size = min(len(indices), 4000)
        if len(indices) < min_samples:
            print(f"  警告：类别 {label} 样本不足（{len(indices)}），已跳过")
            continue
            
        print(f"  {label}: 抽样 {sample_size}/{len(indices)}")
        sampled = np.random.choice(indices, size=sample_size, replace=False)
        sampled_indices.extend(sampled.tolist())
    
    sampled_data = [valid_data[i] for i in sampled_indices]
    print(f"\n📦 抽样后数据量: {len(sampled_data)}")

    # ---------------------- 数据集划分 ----------------------
    print("\n✂️ 正在划分数据集...")
    labels = [extract_label(next(msg["content"] for msg in item["messages"] if msg["role"] == "assistant")) 
             for item in sampled_data]

    # 动态调整参数
    test_size = 0.2 if len(sampled_data) >= 10 else 0.1
    stratify = labels if len(set(labels)) > 1 else None
    
    try:
        train_data, test_data = train_test_split(
            sampled_data,
            test_size=test_size,
            stratify=stratify,
            random_state=42
        )
    except ValueError as e:
        print(f"⚠️ 分层划分失败: {str(e)}")
        print("  尝试非分层划分...")
        train_data, test_data = train_test_split(
            sampled_data,
            test_size=test_size,
            stratify=None,
            random_state=42
        )

    print(f" 训练集: {len(train_data)} | 测试集: {len(test_data)}")

    # ---------------------- 图像处理 ----------------------
    def process_and_copy_images(data, target_img_dir):
        """安全图像处理器"""
        processed_data = []
        for item in data:
            new_images = []
            for img_name in item["images"]:
                src_path = os.path.join(original_img_dir, img_name)
                dst_path = os.path.join(target_img_dir, img_name)
                
                try:
                    with Image.open(src_path) as img:
                        # 中心裁剪
                        w, h = img.size
                        crop_box = (
                            (w - 224) // 2,
                            (h - 224) // 2,
                            (w + 224) // 2,
                            (h + 224) // 2
                        )
                        cropped = img.crop(crop_box)
                        
                        if cropped.mode != 'RGB':
                            cropped = cropped.convert('RGB')
                            
                        cropped.save(dst_path)
                        new_images.append(dst_path)
                except Exception as e:
                    print(f"⚠️ 图像处理失败: {img_name} -> {str(e)}")
                    continue
            
            if new_images:
                processed_data.append({
                    "messages": [
                        {
                            "role": msg["role"],
                            "content": msg["content"].replace("\n", " ").strip()
                        } for msg in item["messages"]
                    ],
                    "images": new_images
                })
        return processed_data

    print("\n🖼️ 正在处理图像...")
    print("  训练集图像处理中...")
    train_processed = process_and_copy_images(train_data, train_img_dir)
    print("  测试集图像处理中...")
    test_processed = process_and_copy_images(test_data, test_img_dir)

    # ---------------------- 数据保存 ----------------------
    def save_jsonl(data, path):
        with open(path, 'w') as f:
            for item in data:
                # 宽松格式验证
                try:
                    assert len(item["messages"]) >= 1, "消息数量不足"
                    assert any(msg["role"] == "user" for msg in item["messages"]), "缺少用户消息"
                    assert any(msg["role"] == "assistant" for msg in item["messages"]), "缺少助手消息"
                except AssertionError as e:
                    print(f"⚠️ 数据格式异常: {e}")
                    continue
                    
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print("\n💾 正在保存数据集...")
    save_jsonl(train_processed, os.path.join(output_dir, "train.jsonl"))
    save_jsonl(test_processed, os.path.join(output_dir, "test.jsonl"))

    # ---------------------- 最终报告 ----------------------
    print("\n" + "="*40)
    print("🏁 处理完成！最终统计:")
    print(f" - 原始数据量: {len(raw_data)}")
    print(f" - 有效数据量: {len(formatted_data)}")
    print(f" - 训练集数量: {len(train_processed)}")
    print(f" - 测试集数量: {len(test_processed)}")
    
    if train_processed:
        sample = train_processed[0]
        print("\n🔍 示例样本:")
        print("用户消息:", sample["messages"][0]["content"][:80] + "...")
        print("助手消息:", sample["messages"][1]["content"][:80] + "...")
        print("图像路径:", sample["images"][0])
    else:
        print("\n⚠️ 警告：训练集为空！")

if __name__ == "__main__":
    process_galaxy_data()